In [2]:
import sys, torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn as nn
sys.path.append('../')
sys.path.append('../../src/fluprofiler/')
from utilities import convert_Pass2tensor, generate_matrix, load_embedding, print_exams
from torch.utils.data import Dataset, DataLoader
from DMS_utilities import fasta_to_dict

In [3]:
def load_crick(path, suffix, id_cols, keep_cols, rename_map, subset=False):
    df = pd.read_csv(path)

    # 只对存在的 id 列加后缀（以防有的文件列不全）
    existing_id_cols = [c for c in id_cols if c in df.columns]
    df[existing_id_cols] = df[existing_id_cols].add(f'_{suffix}')

    if subset:
        # 只选文件里真正存在的列，避免 KeyError
        existing_keep_cols = [c for c in keep_cols if c in df.columns]
        df = df[existing_keep_cols]

        # 只重命名实际存在的列
        effective_rename = {k: v for k, v in rename_map.items() if k in df.columns}
        if effective_rename:
            df = df.rename(columns=effective_rename)

    return df

class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]


In [7]:
import torch
import pandas as pd

device = torch.device('cuda:0')
id_cols = ['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d']
keep_cols = id_cols + ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'serumName', 'virusName',
                        'serumDate', 'virusDate','serumType', 'serumIslID', 'virusIslID', 'HI_Dist']
rename_map = {'serumType': 'Type', 'HI_Dist': 'label',}

Crick_all = pd.read_csv('../../data/reverse_test/processed/All.csv')

In [ ]:
## serum selection
candidate_df = Crick_all
groupby_columns = ['seq_a', 'seq_b', 'serumPassCat']
agg_dict = {'seq_id_a': 'first', 'seq_id_b': 'first', 'label': 'mean', 'serumName': 'first', 'serumIslID': 'first'}
H3_serum = candidate_df.loc[candidate_df['Type'] == 'H3N2'].groupby(groupby_columns).agg(agg_dict).reset_index()
## virus selection
groupby_columns = ['seq_c', 'seq_d', 'virusPassCat']
agg_dict = {'seq_id_c': 'first', 'seq_id_d': 'first', 'label': 'mean', 'virusName': 'first', 'virusIslID': 'first'}
H3_virus = Crick_2025NH.loc[Crick_2025NH['Type'] == 'H3N2'].groupby(groupby_columns).agg(agg_dict).reset_index()

In [5]:
n = len(DMS_sequence)
DMS_serum = pd.DataFrame({'seq_a':DMS_sequence.values(), 'seq_b': [origin_sequence['NA']] * n, 'serumPassCat': ['<EGG>'] * n, 
              'seq_id_a': DMS_sequence.keys(), 'seq_id_b': ['NA'] * n, 'label': [0] * n, 
              'serumName': DMS_sequence.keys(), 'serumIslID': DMS_sequence.keys()})

candidate_serum = pd.concat([DMS_serum, H3_serum])

In [6]:
seq_names = pd.concat([candidate_serum['seq_id_a'], candidate_serum['seq_id_b'], H3_virus['seq_id_c'], H3_virus['seq_id_d']]).unique().tolist()
file_names = ['matrix_' + ID + '.pt' for ID in seq_names]

file_names_DMS = []
file_names_early = []
file_names_2024NH = []
file_names_2024SH = []
file_names_2025NH = []
for f in file_names:
    if '_2024NH' in f:
        file_names_2024NH.append(f.replace('_2024NH', ''))
    elif '_2024SH' in f:
        file_names_2024SH.append(f.replace('_2024SH', ''))
    elif '_2025NH' in f:
        file_names_2025NH.append(f.replace('_2025NH', ''))
    elif '_early' in f:
        file_names_early.append(f.replace('_early', ''))
    else:
        file_names_DMS.append(f)

In [7]:
emb_dict_DMS = load_embedding('./embedding/', map_location=device)

Loading tensor: 100%|██████████| 761/761 [00:29<00:00, 25.95file/s]


In [8]:
emb_dict_DMS.pop('matrix_NA', None)

tensor([[-0.0419, -0.0968, -0.0011,  ..., -0.0937,  0.1378,  0.0086],
        [-0.0515,  0.0827,  0.0144,  ..., -0.0684,  0.0163,  0.0704],
        [ 0.1706,  0.0933, -0.0771,  ..., -0.0365, -0.0071,  0.2442],
        ...,
        [ 0.1159,  0.0475,  0.1970,  ..., -0.1023,  0.0669, -0.0671],
        [ 0.0411, -0.0817,  0.2484,  ..., -0.2033, -0.0025,  0.0006],
        [ 0.0758, -0.0845,  0.1493,  ..., -0.2149,  0.0614,  0.0020]],
       device='cuda:0')

In [9]:
class Emb2Vec(nn.Module):
    def __init__(self, model):
        super(Emb2Vec, self).__init__()
        self.matrix_dropout = model.matrix_dropout
        self.matrix_pooler = model.matrix_pooler
        self.linear = model.linear[0][0]
        self.Gelu = model.linear[0][1]
    def forward(self, embedding):
        embedding = self.matrix_dropout(embedding)
        seq_vector = self.matrix_pooler(embedding)
        seq_vector = self.linear(seq_vector)
        return seq_vector
    
def run_emb2vec_on_dict(emb_dict, emb2vec, device='cpu'):
    emb2vec = emb2vec.to(device)
    emb2vec.eval()

    vec_dict = {}
    for k, emb in emb_dict.items():
        emb = emb.to(device)
        if emb.dim() == 2:   # (566, 2560) -> (1, 566, 2560)
            emb = emb.unsqueeze(0)

        with torch.no_grad():
            vec = emb2vec(emb)[0].cpu()   # (D_out,)
        vec_dict[k] = vec
    return vec_dict


In [ ]:
device = torch.device("cuda:0")
model = torch.load('../../trained_model/1.7_Artificial_back/2025-08-19_17-43-32.pth', weights_only=False, map_location=device)
embedding_pooler = Emb2Vec(model)
vec_dict = run_emb2vec_on_dict(emb_dict_DMS, embedding_pooler, device=device)

In [22]:
vec_dict.keys()

dict_keys(['matrix_K18E', 'matrix_I19T', 'matrix_I19X', 'matrix_N22D', 'matrix_D23N', 'matrix_D23X', 'matrix_T28V', 'matrix_P37S', 'matrix_G39X', 'matrix_T40K', 'matrix_I41V', 'matrix_T44X', 'matrix_I50M', 'matrix_N54X', 'matrix_T56I', 'matrix_N61D', 'matrix_S62Y', 'matrix_I64A', 'matrix_I64M', 'matrix_G65S', 'matrix_K66V', 'matrix_N69G', 'matrix_N69K', 'matrix_S70R', 'matrix_I83M', 'matrix_A85S', 'matrix_L86I', 'matrix_N97S', 'matrix_K98N', 'matrix_K98T', 'matrix_K98X', 'matrix_D101E', 'matrix_N110C', 'matrix_N110Y', 'matrix_S112I', 'matrix_A122V', 'matrix_V128X', 'matrix_S131A', 'matrix_D138I', 'matrix_D138S', 'matrix_S140R', 'matrix_N142T', 'matrix_K156M', 'matrix_S159F', 'matrix_S159P', 'matrix_L173I', 'matrix_L173S', 'matrix_N174X', 'matrix_I176K', 'matrix_M184K', 'matrix_P185Q', 'matrix_Q189H', 'matrix_Q189R', 'matrix_D191E', 'matrix_T203S', 'matrix_D204N', 'matrix_D204X', 'matrix_K205N', 'matrix_K205X', 'matrix_N206S', 'matrix_F208L', 'matrix_F208V', 'matrix_S209Y', 'matrix_S215

In [17]:
import numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go

keys = list(vec_dict.keys())
X = np.stack([vec_dict[k] for k in keys])  # (N, 256)

pca_30 = PCA(n_components=30, random_state=42)
X_30 = pca_30.fit_transform(X)
print("Explained variance (30 PCs):", float(pca_30.explained_variance_ratio_.sum()))

pca_2 = PCA(n_components=2, random_state=42)
X_2d = pca_2.fit_transform(X_30)

target_key = "matrix_HA"
is_target = np.array([k == target_key for k in keys])

fig = go.Figure()

# 先加“其它点”trace
fig.add_trace(go.Scatter(
    x=X_2d[~is_target, 0],
    y=X_2d[~is_target, 1],
    mode="markers",
    name="others",
    text=np.array(keys)[~is_target],     # hover 显示 key
    hovertemplate="%{text}<extra></extra>",
    marker=dict(size=6, opacity=0.8),
))

# 再加“matrix_HA”trace（红色，最后加=显示在最上层）
if is_target.any():
    fig.add_trace(go.Scatter(
        x=X_2d[is_target, 0],
        y=X_2d[is_target, 1],
        mode="markers",
        name=target_key,
        text=np.array(keys)[is_target],
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=10, color="red", opacity=1.0, line=dict(width=1, color="black")),
    ))
else:
    print(f"Warning: key '{target_key}' not found in vec_dict.")

fig.update_layout(
    title="2D embedding (PCA) — hover to show key",
    xaxis_title="Dim 1",
    yaxis_title="Dim 2",
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

Explained variance (30 PCs): 0.8367484211921692


In [13]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
import plotly.graph_objects as go

keys = list(vec_dict.keys())
X = np.stack([vec_dict[k] for k in keys])  # (N, 256)

pca_30 = PCA(n_components=30, random_state=42)
X_30 = pca_30.fit_transform(X)
print("Explained variance (30 PCs):", float(pca_30.explained_variance_ratio_.sum()))

# -----------------------------
# 1) 在 30D PCA 空间里找离群点（鲁棒 Mahalanobis）
# -----------------------------
mcd = MinCovDet(random_state=42).fit(X_30)
md2 = mcd.mahalanobis(X_30)  # squared Mahalanobis distance，越大越离群

# 阈值：优先用卡方阈值（更“统计学”），没有 scipy 就用分位数兜底
try:
    from scipy.stats import chi2
    thr = chi2.ppf(0.99, df=X_30.shape[1])  # 99% 置信阈值
except Exception:
    thr = np.quantile(md2, 0.99)

is_outlier = md2 > thr
outlier_keys = np.array(keys)[is_outlier]

print(f"Outliers (n={is_outlier.sum()}), threshold={thr:.3f}")
for k, s in sorted(zip(outlier_keys, md2[is_outlier]), key=lambda x: x[1], reverse=True):
    print(f"{k}\tmd2={s:.3f}")

# 如果你只想看“最离群的前 K 个”
K = 20
topk_idx = np.argsort(-md2)[:K]
print(f"\nTop-{K} most outlying points:")
for i in topk_idx:
    print(f"{keys[i]}\tmd2={md2[i]:.3f}")

# -----------------------------
# 2) 继续做 2D 展示（和你原来一致）
# -----------------------------
pca_2 = PCA(n_components=2, random_state=42)
X_2d = pca_2.fit_transform(X_30)

target_key = "matrix_HA"
is_target = np.array([k == target_key for k in keys])

fig = go.Figure()

# others（排除 target 和 outliers，避免重复显示）
mask_others = (~is_target) & (~is_outlier)
fig.add_trace(go.Scatter(
    x=X_2d[mask_others, 0],
    y=X_2d[mask_others, 1],
    mode="markers",
    name="others",
    text=np.array(keys)[mask_others],
    hovertemplate="%{text}<extra></extra>",
    marker=dict(size=6, opacity=0.8),
))

# outliers（单独一层）
if is_outlier.any():
    fig.add_trace(go.Scatter(
        x=X_2d[is_outlier, 0],
        y=X_2d[is_outlier, 1],
        mode="markers",
        name="outliers",
        text=np.array(keys)[is_outlier],
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=10, symbol="x", opacity=1.0, line=dict(width=1)),
    ))

# target（最上层）
if is_target.any():
    fig.add_trace(go.Scatter(
        x=X_2d[is_target, 0],
        y=X_2d[is_target, 1],
        mode="markers",
        name=target_key,
        text=np.array(keys)[is_target],
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=10, color="red", opacity=1.0, line=dict(width=1, color="black")),
    ))
else:
    print(f"Warning: key '{target_key}' not found in vec_dict.")

fig.update_layout(
    title="2D embedding (PCA) — hover to show key (outliers marked as x)",
    xaxis_title="Dim 1",
    yaxis_title="Dim 2",
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()


Explained variance (30 PCs): 0.8367484211921692
Outliers (n=421), threshold=50.892
matrix_S235F	md2=7256.558
matrix_S235X	md2=5303.613
matrix_T26M	md2=4971.358
matrix_K156I	md2=4893.563
matrix_V325I	md2=4829.181
matrix_S235C	md2=4498.788
matrix_S235Y	md2=4409.537
matrix_R224X	md2=3950.440
matrix_V325X	md2=3666.876
matrix_S235P	md2=3585.549
matrix_R224K	md2=3549.052
matrix_A228T	md2=2949.543
matrix_N161S	md2=2937.811
matrix_F208I	md2=2901.822
matrix_S235A	md2=2865.467
matrix_T151K	md2=2723.740
matrix_F208V	md2=2581.149
matrix_F211Y	md2=2523.137
matrix_K156L	md2=2482.661
matrix_T151A	md2=2445.412
matrix_K156V	md2=2434.341
matrix_M336X	md2=2330.719
matrix_V59I	md2=2176.740
matrix_K205N	md2=2006.915
matrix_T344X	md2=2004.591
matrix_N161G	md2=1929.516
matrix_T183X	md2=1814.931
matrix_S172H	md2=1804.636
matrix_T151N	md2=1798.261
matrix_A154X	md2=1797.639
matrix_N262X	md2=1790.028
matrix_S25N	md2=1753.284
matrix_R224I	md2=1714.131
matrix_V313X	md2=1694.093
matrix_S25X	md2=1679.746
matrix_K156

In [14]:
import re
from collections import defaultdict, Counter

def summarize_sites(names, prefix="matrix_", top_n=20, min_count=1, ignore_X=False):
    """
    names: 例如 outlier_keys 或 keys
    prefix: key 前缀（默认 matrix_）
    ignore_X: 是否忽略 'X' 这类未知/终止标记（按你数据含义决定）
    """
    # 兼容：matrix_S235F / matrix_R224X / matrix_E292N ...
    pat = re.compile(rf"^{re.escape(prefix)}([A-Z])(\d+)([A-Z\*X])$")

    site_ref = defaultdict(set)         # pos -> {refAA}
    site_alt_counter = defaultdict(Counter)  # pos -> Counter({altAA: count})
    bad = []

    for name in names:
        m = pat.match(name)
        if not m:
            bad.append(name)
            continue
        ref, pos, alt = m.group(1), int(m.group(2)), m.group(3)
        if ignore_X and alt == "X":
            continue
        site_ref[pos].add(ref)
        site_alt_counter[pos][alt] += 1

    # 汇总
    rows = []
    for pos, alt_ct in site_alt_counter.items():
        total = sum(alt_ct.values())
        if total < min_count:
            continue
        refs = "".join(sorted(site_ref[pos]))  # 理论上应只有1个ref；如果出现多个，说明命名/对齐可能混了
        rows.append((total, pos, refs, alt_ct))

    rows.sort(reverse=True, key=lambda x: x[0])

    print(f"Parsed mutations: {len(names)-len(bad)} / {len(names)} (unparsed={len(bad)})")
    if bad:
        print("Examples of unparsed keys:", bad[:10])

    print("\nHigh-frequency sites:")
    for total, pos, refs, alt_ct in rows[:top_n]:
        alt_str = ", ".join([f"{aa}:{cnt}" for aa, cnt in alt_ct.most_common()])
        print(f"pos {pos} (ref={refs})  n={total}  alts=[{alt_str}]")

    return rows

# ===== 用法 1：只统计离群点里高频位点 =====
# outlier_keys = np.array(keys)[is_outlier]  # 你之前已经有了就直接用
rows_outliers = summarize_sites(outlier_keys, top_n=30, min_count=5, ignore_X=False)

# ===== 用法 2：统计全体 keys 里高频位点 =====
# rows_all = summarize_sites(keys, top_n=30, min_count=10, ignore_X=False)


Parsed mutations: 421 / 421 (unparsed=0)

High-frequency sites:
pos 202 (ref=A)  n=9  alts=[D:1, R:1, V:1, G:1, E:1, I:1, S:1, N:1, X:1]
pos 208 (ref=F)  n=8  alts=[L:1, V:1, I:1, S:1, N:1, T:1, X:1, Y:1]
pos 160 (ref=S)  n=8  alts=[D:1, R:1, G:1, K:1, I:1, N:1, T:1, X:1]
pos 110 (ref=N)  n=7  alts=[C:1, Y:1, D:1, H:1, S:1, X:1, K:1]
pos 156 (ref=K)  n=7  alts=[M:1, R:1, L:1, V:1, T:1, X:1, I:1]
pos 64 (ref=I)  n=6  alts=[A:1, M:1, R:1, V:1, T:1, X:1]
pos 66 (ref=K)  n=6  alts=[V:1, G:1, E:1, I:1, X:1, D:1]
pos 69 (ref=N)  n=6  alts=[G:1, V:1, X:1, E:1, D:1, Y:1]
pos 174 (ref=N)  n=6  alts=[X:1, S:1, K:1, D:1, H:1, R:1]
pos 176 (ref=I)  n=6  alts=[K:1, T:1, X:1, E:1, A:1, R:1]
pos 206 (ref=N)  n=6  alts=[S:1, T:1, X:1, V:1, G:1, D:1]
pos 344 (ref=T)  n=6  alts=[I:1, S:1, N:1, X:1, P:1, A:1]
pos 235 (ref=S)  n=6  alts=[X:1, F:1, P:1, A:1, C:1, Y:1]
pos 204 (ref=D)  n=5  alts=[N:1, X:1, E:1, G:1, Y:1]
pos 26 (ref=T)  n=5  alts=[A:1, M:1, K:1, S:1, X:1]
pos 94 (ref=G)  n=5  alts=[V:1, X:1

In [15]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# rows_outliers: summarize_sites(...) 的返回值
top_k = 25
rows = rows_outliers[:top_k]

# -------------------------
# 1) 高频位点：频次柱状图
# -------------------------
df_site = pd.DataFrame({
    "pos": [r[1] for r in rows],
    "ref": [r[2] for r in rows],
    "n":   [r[0] for r in rows],
})
df_site["label"] = df_site["pos"].astype(str) + " (" + df_site["ref"] + ")"
df_site = df_site.sort_values("n", ascending=False)

fig1 = px.bar(
    df_site,
    x="label", y="n",
    hover_data={"pos": True, "ref": True, "n": True},
    title=f"High-frequency mutation sites (Top {top_k})",
)
fig1.update_layout(
    xaxis_title="Site (ref)",
    yaxis_title="Count",
    xaxis_tickangle=-45,
)
fig1.show()

# -------------------------
# 2) 每个位点：有哪些突变（堆叠柱状图）
# -------------------------
records = []
for total, pos, ref, alt_ct in rows:
    for alt, cnt in alt_ct.items():
        records.append({"pos": pos, "ref": ref, "alt": alt, "count": cnt})

df_alt = pd.DataFrame(records)
df_alt["label"] = df_alt["pos"].astype(str) + " (" + df_alt["ref"] + ")"

# 保持 x 轴顺序与 df_site 一致
order = df_site["label"].tolist()
df_alt["label"] = pd.Categorical(df_alt["label"], categories=order, ordered=True)

# 让 X（未知/终止）尽量排到最后（可按需改）
alts = sorted(df_alt["alt"].unique(), key=lambda a: (a == "X", a))

fig2 = go.Figure()
for a in alts:
    sub = df_alt[df_alt["alt"] == a]
    fig2.add_trace(go.Bar(x=sub["label"], y=sub["count"], name=a))

fig2.update_layout(
    barmode="stack",
    title=f"Mutation spectrum per site (stacked, Top {top_k})",
    xaxis_title="Site (ref)",
    yaxis_title="Count",
    xaxis_tickangle=-45,
)
fig2.show()

# -------------------------
# 3) 位点 × 突变氨基酸：热图（更适合看“有哪些突变”）
# -------------------------
pivot = df_alt.pivot_table(index="alt", columns="label", values="count", aggfunc="sum", fill_value=0)

fig3 = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns,
    y=pivot.index,
))
fig3.update_layout(
    title=f"Heatmap of mutations (alt × site, Top {top_k})",
    xaxis_title="Site (ref)",
    yaxis_title="Alt AA",
    xaxis_tickangle=-45,
)
fig3.show()


/tmp/ipykernel_2594768/444019205.py:69: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

